---
**Copyright 2026 Nicola Vessio - Tutti i diritti riservati.**

Questo software (notebook, codice e relativa documentazione) è opera di **Nicola Vessio**
ed è protetto dalle norme vigenti in materia di diritto d'autore (L. 633/1941 e successive
modifiche, Convenzione di Berna, Direttiva 2009/24/CE sulla tutela giuridica dei programmi
per elaboratore).

Sono riservati all'autore tutti i diritti di utilizzazione economica dell'opera, inclusi
a titolo esemplificativo la riproduzione, la distribuzione, la modifica, l'adattamento,
la traduzione e la comunicazione al pubblico. **Ogni uso non espressamente autorizzato
per iscritto dall'autore è vietato.**

Contatto: vessio96@gmail.com

---

# Report delle operazioni MT5 (da file) - Ausilio alla dichiarazione dei redditi

Il presente notebook legge un report di cronistoria esportato da MetaTrader 5
(in formato Excel .xlsx) ed elabora le operazioni in esso contenute, allo scopo di
produrre un report chiaro e ordinato a supporto della dichiarazione dei redditi.

A differenza della versione "live", questo modulo NON si collega al terminale MT5:
lavora su un file già esportato, quindi non richiede che l'applicazione sia aperta.
È adatto a chi non può o non vuole tenere il terminale aperto, o deve elaborare
un estratto fornito da terzi.

## Come esportare ed usare il file da MetaTrader 5

Nel terminale MT5, nella scheda "Cronistoria" (Storico conto), fare clic destro e
scegliere "Report" -> "Foglio di calcolo XML di Office 2007". Questa voce genera un
file con estensione .xlsx, che è il file da fornire a questo notebook.

Nota: viene accettato SOLO il formato .xlsx. Il formato "Report HTML" non è supportato.

Per il corretto funzionamento, *il file generato da MetaTrader5 (ReportHistory...xlsx) deve essere NELLA STESSA CARTELLA di questo notebook.*

## Conti supportati

Lo strumento elabora il report di un conto MetaTrader 5, sia esso un conto operativo
diretto sia un conto in copytrading, presso qualsiasi broker.

## Funzionalità

- Lettura del report .xlsx esportato da MT5
- Estrazione delle operazioni complete, già abbinate da MT5 (apertura + chiusura)
- Separazione tra operazioni di trading e movimenti di cassa (depositi/prelievi)
- Lettura automatica dei dati del conto dall'intestazione del file
- Calcolo del profitto netto dei costi del broker (commissioni + swap)
- Produzione di un report Excel personalizzato a più fogli

## Valuta del conto

Gli importi sono espressi nella valuta del conto (tipicamente USD). L'eventuale
conversione in euro ai fini della dichiarazione dei redditi italiana non è gestita
qui ed è a cura dello studio commercialistico.

## Prerequisiti

- Python 3.12 (ambiente Miniconda consigliato)
- Pacchetti esterni: pandas, openpyxl
- Il file .xlsx esportato da MT5 (vedi istruzioni sopra)

## Nota importante

Questo notebook è uno strumento di organizzazione ed esposizione dei dati, non una
consulenza fiscale. I dati vanno verificati e la dichiarazione è gestita dal
commercialista o CAF. Il regime fiscale applicabile, le eventuali compensazioni, la
corretta imputazione di plusvalenze e minusvalenze, la conversione in euro, il quadro RW
e l'imposta di bollo sono di competenza dello studio commercialistico.

---

## LIBRERIE

Tutte le librerie usate nel notebook, raccolte in un punto solo. Questa cella va eseguita per prima: le celle successive danno per scontato che questi import siano già caricati.

In [ ]:
import glob                                                              # individuare i file nella cartella
import os                                                                # percorsi e nomi file

import pandas as pd                                                      # gestione dei dati in tabelle

from datetime import datetime                                            # interpretare le date delle operazioni

from openpyxl import load_workbook                                       # leggere il file MT5
from openpyxl import Workbook                                            # creare il report Excel
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side   # stili del report
from openpyxl.utils import get_column_letter                             # larghezza colonne
from openpyxl.formatting.rule import CellIsRule                          # regola condizionale: colore per segno

---

## IMPORTAZIONE DATI

Il primo passo è portare dentro il notebook il report esportato da MetaTrader 5 e comprenderne la struttura, così da sapere dove si trovano le informazioni utili.

Il report MT5 è un unico foglio di calcolo organizzato in sezioni successive (operazioni, ordini, movimenti). In questa fase individuiamo il file nella cartella, lo apriamo e localizziamo il punto in cui inizia ciascuna sezione: le celle di estrazione successive sapranno così esattamente dove leggere.

### Selezione del report

Il notebook cerca automaticamente i report esportati da MT5 nella cartella in cui si trova (i file che iniziano con "ReportHistory"). Non occorre indicare nomi o percorsi: basta collocare il file .xlsx lì dentro.

**Non rinominare** il file esportato da MT5: deve mantenere il nome originale che inizia con "ReportHistory", altrimenti il notebook non riesce a individuarlo.

Se nella cartella è presente un solo report, viene usato direttamente. Se ce n'è più di uno (ad esempio conti su broker diversi), il notebook chiede quale analizzare: viene mostrato l'elenco numerato e si digita il numero corrispondente.

> **Nota per VS Code:** quando viene richiesto il numero del report, la casella in cui digitare compare **in alto**, nella barra dei comandi in cima alla finestra, non sotto la cella.

In [ ]:
# Elenchiamo i report trovati nella cartella (file "ReportHistory*.xlsx")
file_trovati = sorted(glob.glob("ReportHistory*.xlsx"))

if not file_trovati:
    raise SystemExit(
        "Nessun file trovato. Collocare il report .xlsx esportato da MT5 "
        "(nome originale tipo 'ReportHistory-XXXXXXXX.xlsx', da NON rinominare) "
        "nella cartella del notebook."
    )

# Se c'e' un solo report, lo usiamo direttamente.
# Se ce n'e' piu' d'uno, chiediamo quale analizzare (l'esecuzione si ferma in attesa).
if len(file_trovati) == 1:
    percorso_file = file_trovati[0]
else:
    print("Trovati piu' report nella cartella:")
    for i, f in enumerate(file_trovati, start=1):
        print(f"  {i}. {f}")

    # Chiediamo il numero finche' non ne arriva uno valido
    while True:
        scelta = input(f"Quale report analizzare? (1-{len(file_trovati)}): ")
        if scelta.isdigit() and 1 <= int(scelta) <= len(file_trovati):
            percorso_file = file_trovati[int(scelta) - 1]
            break
        print(f"  Scelta non valida: digitare un numero da 1 a {len(file_trovati)}.")

print(f"Report selezionato: {percorso_file}")

### Lettura del file e sezioni

Il report MT5 è un unico foglio diviso in sezioni, ciascuna introdotta da una riga-titolo. Qui il foglio viene caricato e si individua a quale riga inizia ciascuna sezione, così le celle successive sanno dove leggere:

- **Posizioni**: le operazioni già abbinate da MT5 (apertura + chiusura)
- **Ordini**: gli ordini immessi (non usati nell'analisi)
- **Affari**: da cui si ricavano i movimenti di cassa (depositi, prelievi, performance fee)

In [ ]:
# Apriamo il file scelto nella cella precedente.
# data_only=True legge i valori (non le eventuali formule).
wb = load_workbook(percorso_file, data_only=True)
ws = wb.active   # il report MT5 ha un unico foglio

# Scorriamo la prima colonna alla ricerca delle righe-titolo di sezione,
# salvando il numero di riga in cui inizia ciascuna.
riga_posizioni = None
riga_ordini    = None
riga_affari    = None

for riga in range(1, ws.max_row + 1):
    valore = ws.cell(row=riga, column=1).value
    if valore == "Posizioni":
        riga_posizioni = riga
    elif valore == "Ordini":
        riga_ordini = riga
    elif valore == "Affari":
        riga_affari = riga

# Se non troviamo le sezioni attese, il file non e' un report MT5 valido.
if riga_posizioni is None or riga_affari is None:
    raise SystemExit(
        "Il file non sembra un report di cronistoria MT5 valido "
        "(sezioni 'Posizioni'/'Affari' non trovate). Verificare di aver esportato "
        "il file corretto da MetaTrader 5."
    )

print("Sezioni individuate:")
print(f"  'Posizioni' inizia alla riga {riga_posizioni}")
print(f"  'Ordini'    inizia alla riga {riga_ordini}")
print(f"  'Affari'    inizia alla riga {riga_affari}")

**Nota sull'avviso "UserWarning" di openpyxl**

Durante la lettura del file, Python potrebbe mostrare un avviso giallo simile a:

`UserWarning: Workbook contains no default style, apply openpyxl's default`

Non è un errore e non compromette in alcun modo i dati o il report. Segnala semplicemente che il file esportato da MetaTrader 5 non contiene uno stile grafico predefinito (i report MT5 sono privi di formattazione): la libreria di lettura ne applica uno proprio e prosegue normalmente. L'elaborazione e i calcoli non sono influenzati, l'avviso può essere ignorato.

---

## ESTRAZIONE E LETTURA DEI DATI

Una volta individuata la struttura del file, ne ricaviamo le informazioni che servono all'analisi, tralasciando tutto il resto. Il report MT5 contiene infatti più dati di quelli utili ai fini fiscali: qui selezioniamo solo ciò che conta e lo organizziamo in tabelle ordinate.

In particolare:

- **le operazioni**: i trade completi già abbinati da MT5 (apertura e chiusura di ogni posizione), con profitto, commissioni e swap;
- **i movimenti di cassa**: depositi, prelievi ed eventuali altri movimenti di denaro, tenuti separati dai trade perché non generano plusvalenze o minusvalenze;
- **i dati del conto**: le informazioni anagrafiche (intestatario, broker, numero di conto, valuta) lette dall'intestazione del file, necessarie per identificare il report.

Vengono invece tralasciati i dati non rilevanti per l'analisi, come gli ordini non eseguiti.

### Estrazione delle operazioni

La sezione "Posizioni" contiene le operazioni già abbinate da MT5: ogni riga è un trade completo (apertura + chiusura) con il relativo profitto. Costruiamo un DataFrame con le stesse colonne del modulo live, così il report Excel finale ha struttura identica.

Le colonne del report MT5 sono, nell'ordine:

- **Ora apertura** - data e ora di apertura della posizione
- **Posizione** - identificativo univoco del trade
- **Simbolo** - strumento negoziato (es. XAUUSD)
- **Tipo** - direzione dell'operazione (buy o sell)
- **Volume** - dimensione in lotti
- **Prezzo apertura** - prezzo di ingresso
- **S/L e T/P** - livelli di stop loss e take profit
- **Ora chiusura** - data e ora di chiusura
- **Prezzo chiusura** - prezzo di uscita
- **Commissioni** - costo trattenuto dal broker
- **Swap** - interesse per il mantenimento overnight
- **Profitto** - risultato lordo dei costi

In [ ]:
# Estrazione delle operazioni dai deal di chiusura (sezione "Affari")
# Ogni deal "out" e' una chiusura = un'operazione fiscale (plusvalenza o minusvalenza).
# Leggiamo da qui, non dalle Posizioni, per includere anche le chiusure parziali:
# una posizione chiusa in piu' tranche genera piu' operazioni, ognuna con il suo risultato.
# Colonne Affari: Ora=1, Affare=2, Simbolo=3, Tipo=4, Direzione=5, Volume=6,
#                 Prezzo=7, Ordine=8, Commissioni=9, Spese=10, Swap=11, Profitto=12

prima_riga_affari = riga_affari + 2

operazioni = []
for riga in range(prima_riga_affari, ws.max_row + 1):
    tipo = ws.cell(row=riga, column=4).value
    direzione = ws.cell(row=riga, column=5).value

    # Solo i deal di trading (buy/sell) in chiusura (out)
    if tipo not in ("buy", "sell") or direzione != "out":
        continue

    # Il tipo del deal di chiusura e' invertito rispetto all'operazione:
    # per chiudere un sell si compra (buy) e viceversa. Lo riportiamo all'originale.
    tipo_operazione = "SELL" if tipo == "buy" else "BUY"

    data_chiusura = ws.cell(row=riga, column=1).value
    affare        = ws.cell(row=riga, column=2).value
    simbolo       = ws.cell(row=riga, column=3).value
    volume        = ws.cell(row=riga, column=6).value
    prezzo        = ws.cell(row=riga, column=7).value
    commissione   = ws.cell(row=riga, column=9).value or 0.0
    swap          = ws.cell(row=riga, column=11).value or 0.0
    profitto      = ws.cell(row=riga, column=12).value or 0.0

    profit_netto = profitto + swap + commissione

    operazioni.append({
        "id":            affare,
        "symbol":        simbolo,
        "tipo":          tipo_operazione,
        "volume":        volume,
        "prezzo":        prezzo,
        "data_chiusura": data_chiusura,
        "profit_lordo":  round(profitto, 2),
        "swap":          round(swap, 2),
        "commission":    round(commissione, 2),
        "profit_netto_costi_broker_lordo_imposte_USD": round(profit_netto, 2),
    })

df_trade = pd.DataFrame(operazioni).sort_values("data_chiusura").reset_index(drop=True)

print(f"Operazioni estratte (dai deal di chiusura): {len(df_trade)}")
print(f"Risultato totale (USD): {df_trade['profit_netto_costi_broker_lordo_imposte_USD'].sum():.2f}")

### Estrazione dei movimenti di cassa

Depositi, prelievi e performance fee non compaiono tra le "Posizioni" (che sono solo trade): si trovano nella sezione "Affari", nelle righe di tipo "balance". Sono movimenti di denaro, non operazioni di trading, e vanno tenuti separati perché non generano plusvalenze o minusvalenze.

Ogni movimento viene classificato in una di quattro categorie, in base al segno dell'importo e alla descrizione:

- **Deposito**: importo positivo (versamento di capitale sul conto)
- **Performance fee**: importo negativo con descrizione "split" (il compenso trattenuto dal servizio di copytrading)
- **Prelievo**: importo negativo con descrizione "withdraw" (trasferimento verso il conto bancario)
- **Altro**: importo negativo senza le due parole chiave (movimento da verificare, che non viene comunque perso)

Nella sezione "Affari" i dati utili si leggono da tre colonne: il **Tipo** (colonna 4, dove compare "balance"), il **Profitto** (colonna 12, che contiene l'importo del movimento) e il **Commento** (colonna 14, la descrizione).

In [ ]:
# I dati degli Affari iniziano 2 righe dopo il titolo (titolo + intestazione)
prima_riga_affari = riga_affari + 2

movimenti_cassa = []
for riga in range(prima_riga_affari, ws.max_row + 1):
    tipo = ws.cell(row=riga, column=4).value   # colonna "Tipo"

    # Ci fermiamo quando finiscono i dati: se anche l'ora (col 1) e' vuota, usciamo
    if ws.cell(row=riga, column=1).value is None:
        break

    # Ci interessano SOLO le righe di tipo "balance" (movimenti di cassa)
    if tipo is not None and str(tipo).lower() == "balance":
        ora      = ws.cell(row=riga, column=1).value    # data/ora del movimento
        affare   = ws.cell(row=riga, column=2).value    # id del movimento
        importo  = ws.cell(row=riga, column=12).value   # colonna "Profitto" = importo
        commento = ws.cell(row=riga, column=14).value   # descrizione del movimento

        importo = importo if importo is not None else 0.0
        descrizione = str(commento).lower() if commento is not None else ""

        # Classifichiamo il movimento in base al segno e alla descrizione.
        # L'ordine conta: prima il deposito (positivo), poi le parole chiave.
        # "split" identifica la performance fee del servizio di copytrading.
        if importo > 0:
            categoria = "Deposito"
        elif "split" in descrizione:
            categoria = "Performance fee"
        elif "withdraw" in descrizione:
            categoria = "Prelievo"
        else:
            categoria = "Altro"

        movimenti_cassa.append({
            "id":        affare,
            "data":      ora,
            "importo":   round(importo, 2),
            "categoria": categoria,
            "commento":  commento if commento is not None else "",
        })

# Riepilogo di controllo a schermo, per categoria
print(f"Movimenti di cassa trovati: {len(movimenti_cassa)}")
for m in movimenti_cassa:
    print(f"  {m['data']} | {m['categoria']:16} | {m['importo']:>10.2f} USD | {m['commento']}")

### Lettura dei dati del conto

I dati anagrafici del conto (intestatario, broker, numero, valuta) si trovano nell'intestazione del report, nelle prime righe del foglio. Li leggiamo per identificare il conto nel report Excel finale.

Il numero di conto è scritto su una riga unica insieme ad altri dati, nel formato `NUMERO (VALUTA, SERVER, TIPO, MODALITA)` - ad esempio `12345678 (USD, BrokerServer-Live, real, Hedge)`. La riga viene quindi scomposta nei singoli campi.

In [ ]:
# Leggiamo i valori grezzi dall'intestazione (colonna 4)
nome_intestatario = ws.cell(row=2, column=4).value or "n/d"
riga_conto        = ws.cell(row=3, column=4).value or ""
societa           = ws.cell(row=4, column=4).value or "n/d"

# Spacchettiamo la riga del conto. Formato: "NUMERO (VALUTA, SERVER, TIPO, MODALITA)"
numero_conto = "n/d"
valuta       = "n/d"
server       = "n/d"
tipo_conto   = "n/d"
modalita     = "n/d"

if riga_conto:
    if "(" in riga_conto:
        # Prima della parentesi: il numero conto
        numero_conto = riga_conto.split("(")[0].strip()
        # Tra parentesi: valuta, server, tipo, modalita' (separati da virgola)
        dentro_parentesi = riga_conto.split("(")[1].rstrip(")")
        parti = [p.strip() for p in dentro_parentesi.split(",")]
        if len(parti) >= 1:
            valuta = parti[0]
        if len(parti) >= 2:
            server = parti[1]
        if len(parti) >= 3:
            tipo_conto = parti[2]
        if len(parti) >= 4:
            modalita = parti[3]
    else:
        numero_conto = riga_conto.strip()

# Raccogliamo tutto in un dizionario (come "info" del modulo live)
info_conto = {
    "intestatario": nome_intestatario,
    "societa":      societa,
    "numero_conto": numero_conto,
    "server":       server,
    "valuta":       valuta,
    "tipo_conto":   tipo_conto,
    "modalita":     modalita,
}

print("Dati del conto letti dal file:")
for chiave, valore in info_conto.items():
    print(f"  {chiave}: {valore}")

---

## GENERAZIONE DEL REPORT EXCEL

L'ultima fase produce il documento vero e proprio: un file Excel ordinato e leggibile, pensato per essere consegnato allo studio commercialistico come supporto alla dichiarazione. Non sostituisce il documento fiscale ufficiale, ma organizza i dati grezzi in una forma chiara e verificabile.

Il report è composto da quattro fogli, ciascuno con uno scopo preciso:

- **Operazioni**: l'elenco dettagliato di ogni trade, con risultato lordo, costi (swap e commissioni) e risultato netto, raggruppati per mese;
- **Riepilogo**: i dati del conto e i totali fiscali complessivi, calcolati con formule Excel vive;
- **Movimenti di cassa**: depositi, prelievi e performance fee, tenuti separati dai trade perché non generano plusvalenze o minusvalenze;
- **Guida e glossario**: le spiegazioni dei termini, per rendere il report comprensibile anche a chi non opera nel trading.

Le celle che seguono definiscono prima lo stile grafico comune, poi costruiscono un foglio alla volta, e infine salvano il file.

### Stile/setup del report

Questa cella definisce, una sola volta, l'identità visiva condivisa da tutti i fogli del report: la palette dei colori, i caratteri e le funzioni di formattazione riutilizzabili. Concentrare qui lo stile permette di modificarlo in un unico punto e mantenerlo coerente su tutto il documento.

L'aspetto del report segue queste scelte:

- **Intestazioni**: viola notte con testo oro, in maiuscolo e grassetto, per distinguerle nettamente dai dati;
- **Record**: carattere ampio (Bahnschrift) su righe alterne bianche e grigio chiaro, per una lettura riposante;
- **Colonne**: adattate automaticamente alla lunghezza del contenuto, così nessun dato risulta troncato;
- **Allineamento**: contenuto centrato in ogni cella, per un aspetto ordinato e uniforme.

In [ ]:
# Font del report (Bahnschrift e' preinstallato su Windows)
FONT = "Bahnschrift"

# Palette dei colori
VIOLA    = "3C1361"   # viola notte - intestazioni e bande di sezione
ORO      = "D4AF37"   # oro - testo sulle intestazioni
GRIGIO   = "F5F5F7"   # grigio chiaro - righe alterne (zebra)
NERO     = "1A1A1A"   # testo dei record
BORDO_C  = "E5E5EA"   # bordi sottili
VERDE    = "1E7B34"   # risultati positivi
ROSSO    = "C0392B"   # risultati negativi (rosso pieno)

# Scelta dei font
font_head = Font(name=FONT, size=14, bold=True, color=ORO)   # intestazioni: grandi, grassetto
font_norm = Font(name=FONT, size=12, color=NERO)             # record
font_bold = Font(name=FONT, size=12, bold=True, color=NERO)  # record in risalto (totali)

# Riempimenti, bordi, allineamento
fill_head  = PatternFill("solid", fgColor=VIOLA)
fill_zebra = PatternFill("solid", fgColor=GRIGIO)
bordo      = Border(*[Side(style="thin", color=BORDO_C)] * 4)
centro     = Alignment(horizontal="center", vertical="center")
sinistra   = Alignment(horizontal="left", vertical="center", wrap_text=True)   # testi lunghi: paragrafi e definizioni

# Riga di intestazioni colonne: viola/oro, maiuscole, centrate
def scrivi_intestazioni(ws, intestazioni, riga=1):
    for col, testo in enumerate(intestazioni, start=1):
        c = ws.cell(row=riga, column=col, value=str(testo).upper())
        c.font = font_head
        c.fill = fill_head
        c.alignment = centro
        c.border = bordo

# Titolo di sezione su banda viola, esteso su n_colonne
def banda_titolo(ws, riga, testo, n_colonne):
    for col in range(1, n_colonne + 1):
        c = ws.cell(row=riga, column=col)
        c.fill = fill_head
        c.border = bordo
        if col == 1:
            c.value = str(testo).upper()
            c.font = font_head
            c.alignment = centro

# Adatta la larghezza di ogni colonna al contenuto piu' lungo (autofit simulato).
# fattore: compensa il font ampio e proporzionale. I limiti minimo/massimo
# evitano colonne troppo strette o troppo larghe (il n° è espresso in "caratteri", non pixel).
def autofit_colonne(ws, fattore=1.5, minimo=8, massimo=40):
    for col in ws.columns:
        lettera = get_column_letter(col[0].column)
        max_len = max((len(str(c.value)) for c in col if c.value is not None), default=0)
        ws.column_dimensions[lettera].width = max(minimo, min(massimo, (max_len + 2) * fattore))

# Colora una colonna in base al segno: verde se >= 0, bordeaux se < 0.
# Usa la formattazione condizionale di Excel, cosi' funziona anche sulle formule
# (il colore si aggiorna col valore calcolato, non e' fissato alla scrittura).
def colora_per_segno(ws, colonna, prima_riga, ultima_riga):
    rng = f"{colonna}{prima_riga}:{colonna}{ultima_riga}"
    ws.conditional_formatting.add(rng,
        CellIsRule(operator="lessThan", formula=["0"],
                   font=Font(name=FONT, size=12, color="FF" + ROSSO)))
    ws.conditional_formatting.add(rng,
        CellIsRule(operator="greaterThanOrEqual", formula=["0"],
                   font=Font(name=FONT, size=12, color="FF" + VERDE)))

### Preparazione dei dati

Prima di costruire i fogli, prepariamo i dati che serviranno al report: la tabella dei movimenti di cassa, l'intervallo di date coperto e l'anno d'imposta di riferimento.

L'anno d'imposta è ricavato dall'anno di **chiusura** delle operazioni, perché è la chiusura a determinare quando la plusvalenza o minusvalenza si realizza fiscalmente. Un'operazione aperta a fine dicembre e chiusa a gennaio appartiene quindi all'anno successivo.

In [ ]:
# Movimenti di cassa in tabella (lista di dizionari -> DataFrame)
df_cassa = pd.DataFrame(movimenti_cassa)

# Ultima riga con dati nel foglio Movimenti (intestazione in riga 1, dati da riga 2)
ultima_riga_cassa = len(df_cassa) + 1

# Periodo coperto dal report: dal primo all'ultimo evento.
# Consideriamo tutte le date (apertura e chiusura dei trade, piu' i movimenti),
# cosi' il periodo include anche prelievi o fee successivi all'ultimo trade.
date_report = []
if not df_trade.empty:
    date_report += [str(d) for d in df_trade["data_chiusura"]]
if not df_cassa.empty:
    date_report += [str(d) for d in df_cassa["data"]]

if date_report:
    periodo_dal = min(date_report)[:10]
    periodo_al  = max(date_report)[:10]
else:
    periodo_dal = periodo_al = "n/d"

# Anno d'imposta: dall'anno delle date di CHIUSURA (evento fiscalmente rilevante)
if not df_trade.empty:
    anni_chiusura = sorted({str(d)[:4] for d in df_trade["data_chiusura"]})
else:
    anni_chiusura = []

if len(anni_chiusura) == 1:
    anno_fiscale = anni_chiusura[0]
elif len(anni_chiusura) > 1:
    anno_fiscale = " / ".join(anni_chiusura)   # caso raro: chiusure su piu' anni
else:
    anno_fiscale = "n/d"

print(f"Periodo coperto: dal {periodo_dal} al {periodo_al}")
print(f"Anno d'imposta (da date di chiusura): {anno_fiscale}")
print(f"Movimenti di cassa: {len(df_cassa)}")

### Foglio 1 - Operazioni

Il primo foglio elenca ogni operazione chiusa nell'anno, ricavata dai movimenti di chiusura del conto. Ogni chiusura, anche parziale, è un'operazione a sé con la propria plusvalenza o minusvalenza. Le operazioni sono raggruppate per mese, con un subtotale a fine di ciascuno. I risultati positivi appaiono in verde, quelli negativi in rosso.

In [ ]:
# Creiamo il file Excel e il primo foglio
wb = Workbook()
wb.calculation.fullCalcOnLoad = True
ws1 = wb.active
ws1.title = "1. Operazioni"

intest = [
    "N.", "ID", "Data", "Coppia / Strumento", "Tipo", "Volume", "Prezzo",
    "Risultato lordo (USD)", "Swap (USD)", "Commissione (USD)", "Risultato netto (USD)",
]
scrivi_intestazioni(ws1, intest, riga=1)

# Nomi dei mesi in italiano (per le righe di subtotale)
NOMI_MESI = {
    "01": "Gennaio", "02": "Febbraio", "03": "Marzo", "04": "Aprile",
    "05": "Maggio", "06": "Giugno", "07": "Luglio", "08": "Agosto",
    "09": "Settembre", "10": "Ottobre", "11": "Novembre", "12": "Dicembre",
}

# Sfondo e testo delle righe di subtotale mensile
GRIGIO_SUB = "E5E5EA"
fill_sub = PatternFill("solid", fgColor=GRIGIO_SUB)
font_sub = Font(name=FONT, size=12, bold=True, color=VIOLA)

def scrivi_subtotale(ws, riga, mese_str, primo, ultimo):
    nome = f"{NOMI_MESI[mese_str[5:7]]} {mese_str[:4]}"
    for col in range(1, 12):
        c = ws.cell(row=riga, column=col)
        c.fill = fill_sub
        c.border = bordo
    c = ws.cell(row=riga, column=1, value=f"TOTALE {nome.upper()}")
    c.font = font_sub
    c.alignment = centro
    for lettera, num in [("H", 8), ("I", 9), ("J", 10), ("K", 11)]:
        c = ws.cell(row=riga, column=num, value=f"=SUM({lettera}{primo}:{lettera}{ultimo})")
        c.font = font_sub
        c.alignment = centro
        c.number_format = "#,##0.00"

# Ordiniamo per data di chiusura e ricaviamo il mese
df_op = df_trade.sort_values("data_chiusura").reset_index(drop=True)
df_op["mese"] = df_op["data_chiusura"].astype(str).str[:7]

riga = 2
n = 1
mese_corrente = None
primo_riga_mese = 2

for _, r in df_op.iterrows():
    mese = r["mese"]
    if mese_corrente is not None and mese != mese_corrente:
        scrivi_subtotale(ws1, riga, mese_corrente, primo_riga_mese, riga - 1)
        riga += 1
        primo_riga_mese = riga
    mese_corrente = mese

    zebra = (n % 2 == 0)
    valori = [
        n, r["id"], str(r["data_chiusura"]), r["symbol"], r["tipo"],
        r["volume"], r["prezzo"],
        r["profit_lordo"], r["swap"], r["commission"],
        f"=H{riga}+I{riga}+J{riga}",
    ]
    for col, v in enumerate(valori, start=1):
        c = ws1.cell(row=riga, column=col, value=v)
        c.font = font_norm
        c.border = bordo
        c.alignment = centro
        if col in (8, 9, 10, 11):
            c.number_format = "#,##0.00"
        if zebra:
            c.fill = fill_zebra
    riga += 1
    n += 1

if mese_corrente is not None:
    scrivi_subtotale(ws1, riga, mese_corrente, primo_riga_mese, riga - 1)

ultima_riga_op = riga

colora_per_segno(ws1, "H", 2, ultima_riga_op)
colora_per_segno(ws1, "K", 2, ultima_riga_op)

ws1.freeze_panes = "A2"
autofit_colonne(ws1)

print(f"Foglio Operazioni creato: {n - 1} operazioni.")

### Foglio 2 - Riepilogo

Il quadro d'insieme dell'anno: dati del conto, totali fiscali e movimenti di cassa. I totali sono formule vive che sommano dagli altri fogli, contando solo le operazioni e non i subtotali. Plusvalenze e minusvalenze sono separate; l'ultima riga riporta il risultato netto dopo la performance fee del servizio di copytrading.

In [ ]:
ws2 = wb.create_sheet("2. Riepilogo")

# Banda titolo di sezione (viola/oro, su 2 colonne)
def banda_sezione(ws, riga, testo):
    for col in (1, 2):
        c = ws.cell(row=riga, column=col)
        c.fill = fill_head
        c.border = bordo
    c = ws.cell(row=riga, column=1, value=str(testo).upper())
    c.font = font_head
    c.alignment = centro

# --- Blocco 1: dati del conto ---
banda_sezione(ws2, 1, "Dati del conto")
dati_conto = [
    ("Intestatario",   info_conto["intestatario"]),
    ("Broker",         info_conto["societa"]),
    ("Numero conto",   info_conto["numero_conto"]),
    ("Server",         info_conto["server"]),
    ("Valuta",         info_conto["valuta"]),
    ("Tipo conto",     info_conto["tipo_conto"]),
    ("Modalita'",      info_conto["modalita"]),
    ("Anno d'imposta", anno_fiscale),
    ("Periodo",        f"dal {periodo_dal} al {periodo_al}"),
]
r = 2
for etichetta, valore in dati_conto:
    ce = ws2.cell(row=r, column=1, value=etichetta)
    cv = ws2.cell(row=r, column=2, value=valore)
    ce.font = font_bold; cv.font = font_norm
    ce.border = bordo;   cv.border = bordo
    ce.alignment = centro; cv.alignment = centro
    if r % 2 == 0:
        ce.fill = fill_zebra; cv.fill = fill_zebra
    r += 1

# --- Blocco 2: totali fiscali (formule vive, contano solo le operazioni via N.>0) ---
r += 1
banda_sezione(ws2, r, "Totali fiscali dell'anno")
r += 1
op = "'1. Operazioni'"
cN = f"{op}!A2:A{ultima_riga_op}"
cL = f"{op}!H2:H{ultima_riga_op}"
cS = f"{op}!I2:I{ultima_riga_op}"
cC = f"{op}!J2:J{ultima_riga_op}"
cK = f"{op}!K2:K{ultima_riga_op}"
totali = [
    ("Numero operazioni",       f'=COUNTIF({cN},">0")'),
    ("Operazioni in guadagno",  f'=COUNTIFS({cN},">0",{cK},">0")'),
    ("Operazioni in perdita",   f'=COUNTIFS({cN},">0",{cK},"<0")'),
    ("Risultato lordo totale",  f'=SUMIF({cN},">0",{cL})'),
    ("Totale swap",             f'=SUMIF({cN},">0",{cS})'),
    ("Totale commissioni",      f'=SUMIF({cN},">0",{cC})'),
    ("Plusvalenze",             f'=SUMIFS({cK},{cN},">0",{cK},">0")'),
    ("Minusvalenze",            f'=SUMIFS({cK},{cN},">0",{cK},"<0")'),
    ("Risultato netto totale",  f'=SUMIF({cN},">0",{cK})'),
]
prima_r_totali = r
for etichetta, formula in totali:
    ce = ws2.cell(row=r, column=1, value=etichetta)
    cv = ws2.cell(row=r, column=2, value=formula)
    ce.font = font_bold; cv.font = font_norm
    ce.border = bordo;   cv.border = bordo
    ce.alignment = centro; cv.alignment = centro
    cv.number_format = "#,##0.00"
    if r % 2 == 0:
        ce.fill = fill_zebra; cv.fill = fill_zebra
    r += 1
netto_totale_riga = r - 1   # riga del "Risultato netto totale" (serve al dopo-fee)

# Colore per segno sugli importi (salta numero operazioni + i due conteggi = prime 3 righe)
colora_per_segno(ws2, "B", prima_r_totali + 3, r - 1)

# --- Blocco 3: movimenti di cassa (totali per categoria + risultato dopo fee) ---
r += 1
banda_sezione(ws2, r, "Movimenti di cassa")
r += 1
mv = "'3. Movimenti di cassa'"
cCat = f"{mv}!D2:D{ultima_riga_cassa}"
cImp = f"{mv}!E2:E{ultima_riga_cassa}"
movimenti = [
    ("Totale depositi",        f'=SUMIF({cCat},"Deposito",{cImp})'),
    ("Totale prelievi",        f'=SUMIF({cCat},"Prelievo",{cImp})'),
    ("Totale performance fee", f'=SUMIF({cCat},"Performance fee",{cImp})'),
]
prima_r_mov = r
for etichetta, formula in movimenti:
    ce = ws2.cell(row=r, column=1, value=etichetta)
    cv = ws2.cell(row=r, column=2, value=formula)
    ce.font = font_bold; cv.font = font_norm
    ce.border = bordo;   cv.border = bordo
    ce.alignment = centro; cv.alignment = centro
    cv.number_format = "#,##0.00"
    if r % 2 == 0:
        ce.fill = fill_zebra; cv.fill = fill_zebra
    r += 1
fee_riga = r - 1   # riga del "Totale performance fee"

# Risultato dopo fee = netto totale + performance fee (fee gia' negativa)
ce = ws2.cell(row=r, column=1, value="RISULTATO DOPO FEE")
cv = ws2.cell(row=r, column=2, value=f"=B{netto_totale_riga}+B{fee_riga}")
ce.font = font_bold; cv.font = font_bold
ce.border = bordo;   cv.border = bordo
ce.alignment = centro; cv.alignment = centro
cv.number_format = "#,##0.00"
ce.fill = fill_sub; cv.fill = fill_sub
r += 1

# --- Disclaimer in fondo (grassetto, wrap su piu' righe) ---
r += 1
c = ws2.cell(row=r, column=1,
             value="Tutti gli importi sono in USD. Conversione in EUR, quadro RW e "
                   "imposta di bollo a cura dello studio commercialistico.")
c.font = font_bold
c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
ws2.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
ws2.row_dimensions[r].height = 48

autofit_colonne(ws2)

print("Foglio Riepilogo creato.")

### Foglio 3 - Movimenti di cassa

Il terzo foglio elenca i movimenti di denaro sul conto: depositi, prelievi e performance fee. Sono tenuti separati dalle operazioni di trading perché non generano plusvalenze o minusvalenze: entrano nel calcolo del capitale, non in quello del risultato fiscale.

In [ ]:
ws3 = wb.create_sheet("3. Movimenti di cassa")

intest_cassa = ["N.", "ID", "Data", "Categoria", "Importo (USD)", "Descrizione"]
scrivi_intestazioni(ws3, intest_cassa, riga=1)

# Colore blu per gli importi (movimenti di cassa: neutri, non profitti/perdite)
BLU = "2C5AA0"
font_blu = Font(name=FONT, size=12, color=BLU)

riga = 2
if not df_cassa.empty:
    for n, (_, r_) in enumerate(df_cassa.iterrows(), start=1):
        valori = [n, r_["id"], str(r_["data"]), r_["categoria"], r_["importo"], r_["commento"]]
        for col, v in enumerate(valori, start=1):
            c = ws3.cell(row=riga, column=col, value=v)
            c.border = bordo
            c.alignment = centro
            c.font = font_blu if col == 5 else font_norm   # importo in blu
            if col == 5:
                c.number_format = "#,##0.00"
            if n % 2 == 0:
                c.fill = fill_zebra
        riga += 1

# Disclaimer in fondo (grassetto)
riga += 1
c = ws3.cell(row=riga, column=1,
             value="I movimenti di cassa (depositi, prelievi, performance fee) non sono "
                   "operazioni di trading e non generano plusvalenze o minusvalenze.")
c.font = Font(name=FONT, size=11, bold=True, color=NERO)
c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
ws3.merge_cells(start_row=riga, start_column=1, end_row=riga, end_column=6)

ws3.freeze_panes = "A2"
autofit_colonne(ws3)

print(f"Foglio Movimenti di cassa creato: {len(df_cassa)} movimenti.")

### Foglio 4 - Guida e glossario

Ultimo foglio del report. Contiene due parti: una nota su come leggere i dati e il glossario dei termini, così che il documento sia comprensibile anche a chi non opera sui mercati finanziari.

La nota in cima chiarisce i punti piu' fraintendibili:

- natura delle operazioni: contratti derivati (CFD), senza compravendita fisica del sottostante;
- BUY e SELL come posizioni al rialzo o al ribasso, non acquisti o vendite reali;
- valuta del conto (USD) e adempimenti a carico dello studio commercialistico;
- numero di operazioni (le chiusure) superiore a quello delle posizioni aperte quando alcune sono chiuse in piu' tranche, senza che cambi il risultato netto.

Segue il glossario, che definisce una volta sola ogni termine presente nel report.

In [ ]:
ws4 = wb.create_sheet("Guida e glossario")

banda_titolo(ws4, 1, "COME LEGGERE QUESTO REPORT", 2)
ws4.merge_cells(start_row=1, start_column=1, end_row=1, end_column=2)

paragrafi = [
    'Questo report riguarda operazioni di trading su strumenti finanziari derivati (CFD) '
    'effettuate tramite broker esteri.',
    '"BUY" e "SELL" NON sono acquisti o vendite di beni o valute reali. Sono contratti (CFD) '
    'che replicano la variazione di prezzo di uno strumento: non si possiede mai il bene o la '
    'valuta sottostante. BUY = puntata sul rialzo (posizione lunga); SELL = puntata sul ribasso '
    '(posizione corta). Il risultato è solo un importo in denaro (USD).',
    'FOREX (coppie di valute): strumenti come EURUSD, GBPUSD, USDJPY. Si opera sulla variazione '
    'del tasso di cambio, non sullo scambio fisico delle valute. XAUUSD indica oro/dollaro, '
    'anch\'esso trattato come strumento e mai consegnato fisicamente.',
    'VALUTA: il conto è denominato in dollari USA (USD). Tutti gli importi sono in USD. '
    'La conversione in euro è a cura dello studio commercialistico.',
    'NUMERO DI OPERAZIONI: ogni operazione corrisponde a una chiusura. Quando una posizione '
    'viene chiusa in più momenti (chiusure parziali), ciascuna chiusura è contata come '
    'operazione a sé: per questo il numero di operazioni può risultare superiore al numero '
    'di posizioni aperte. Il risultato netto complessivo non cambia.',
    'ALTRI REDDITI: nel periodo non risultano dividendi, cedole, interessi attivi o altri '
    'redditi finanziari diversi dal risultato delle operazioni di trading.',
    'BROKER ESTERI: quando il broker ha sede estera, gli adempimenti collegati '
    '(quadro RW, imposta di bollo) sono curati dallo studio commercialistico.',
]

def righe_testo(testo, caratteri_per_riga):
    # Quante righe occupa il testo una volta mandato a capo, contando anche
    # gli a-capo espliciti. Stima prudente, cosi' non si taglia su altri PC.
    righe = 0
    for parte in str(testo).split("\n"):
        n = len(parte)
        righe += -(-n // caratteri_per_riga) if n else 1
    return max(1, righe)

r = 3
for p in paragrafi:
    c = ws4.cell(row=r, column=1, value=p)
    c.font = font_norm
    c.alignment = sinistra
    ws4.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
    ws4.row_dimensions[r].height = righe_testo(p, 95) * 16 + 4
    r += 2

r += 1
banda_titolo(ws4, r, "GLOSSARIO DEI TERMINI", 2)
ws4.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
r += 1
scrivi_intestazioni(ws4, ["Termine", "Significato"], riga=r)
r += 1

glossario = [
    ("CFD", "Contratto che replica il prezzo di uno strumento senza possederlo. Risultato in denaro."),
    ("Forex", "Mercato dei tassi di cambio, strumenti espressi come coppie di valute."),
    ("Coppia di valute", "Es. EURUSD: prima valuta 'base', seconda 'quotata'. Si opera sul cambio."),
    ("BUY (posizione lunga)", "Puntata sul rialzo del prezzo. Non è acquisto di beni/valute reali."),
    ("SELL (posizione corta)", "Puntata sul ribasso del prezzo. Non è vendita di beni/valute reali."),
    ("Operazione (trade)", "Singola operazione chiusa (realizzo) rilevata nel periodo."),
    ("ID operazione", "Codice che identifica la singola operazione."),
    ("Volume (lotti)", "Dimensione dell'operazione. Non indica quantità di merce/valuta posseduta."),
    ("Strumento", "Nome dello strumento negoziato (es. EURUSD, GBPUSD, XAUUSD)."),
    ("Risultato lordo", "Esito dell'operazione prima dei costi del broker. In USD."),
    ("Commissione", "Costo trattenuto dal broker sull'operazione. Valore negativo. In USD."),
    ("Swap", "Interesse per il mantenimento della posizione oltre la giornata. Di norma negativo. In USD."),
    ("Risultato netto", "Esito dopo commissioni e swap (costi del broker), al lordo delle imposte. In USD."),
    ("Movimento di cassa", "Deposito o prelievo di denaro sul conto. Non è un'operazione di trading e "
    "non genera plus o minusvalenze. Il deposito è denaro proprio versato dal titolare: non è reddito "
    "imponibile né plusvalenza."),
    ("Performance fee", "Compenso trattenuto dal gestore del segnale di copytrading, calcolato sui risultati del periodo. "
    "Viene addebitato come movimento separato sul conto, di norma all'inizio del mese successivo a quello di riferimento: "
    "la fee visibile a inizio mese si riferisce quindi al mese precedente. Importo in USD. "
    "Il relativo trattamento fiscale è di competenza dello studio commercialistico."),
    ("Plusvalenza", "Guadagno realizzato su un'operazione."),
    ("Minusvalenza", "Perdita realizzata su un'operazione."),
    ("Quadro RW", "Sezione per il monitoraggio delle attività finanziarie estere. A cura dello studio."),
    ("Imposta di bollo", "Imposta annua sui conti oltre soglie di giacenza. A cura dello studio."),
]

for i, (termine, significato) in enumerate(glossario):
    ct = ws4.cell(row=r, column=1, value=termine)
    cs = ws4.cell(row=r, column=2, value=significato)
    ct.font = font_bold
    cs.font = font_norm
    ct.alignment = centro
    cs.alignment = sinistra
    ct.border = bordo
    cs.border = bordo
    if i % 2 == 1:
        ct.fill = fill_zebra
        cs.fill = fill_zebra
    ws4.row_dimensions[r].height = righe_testo(significato, 72) * 16 + 4
    r += 1

ws4.column_dimensions["A"].width = 30
ws4.column_dimensions["B"].width = 85

print(f"Foglio Guida e glossario creato: {len(glossario)} voci di glossario.")

## SALVATAGGIO DEL REPORT

Salva il report Excel finito. Il nome del file viene composto in automatico, così che ogni report sia riconoscibile senza doverlo rinominare a mano:

- il nome del broker, letto dal file ed epurato dalle parole generiche (Limited, Ltd, ecc.) che non lo identificano;
- il periodo coperto (dalla prima all'ultima data presente nel report);
- il suffisso "da-file", che distingue questo modulo da quello che si collega al terminale live.

Il file viene salvato sul Desktop. Nota importante: il report contiene formule vive (i totali del Riepilogo), che vanno calcolate aprendo e salvando il file una volta in Excel prima di inviarlo allo studio commercialistico.

In [ ]:
# --- Nome broker automatico, letto dal file ---
# Usa il nome società completo (es. "TradeMax Global Limited"), togliendo le
# parole generiche finali (Limited, Ltd, ecc.) che non identificano il broker,
# e sostituendo gli spazi con trattini per renderlo adatto a un nome file.
# Se il nome non è disponibile, usa un ripiego generico.
company = str(info_conto["societa"]).strip()

if company and company.lower() != "n/d":
    for generica in ["Limited", "Ltd", "LLC", "Inc", "Incorporated", "Group", "Pty", "S.A.", "SA", "SpA"]:
        company = company.replace(generica, "")
    broker = "-".join(company.split())
    if not broker:
        broker = "BROKER"
else:
    broker = "BROKER"

# Periodo per il nome file: usiamo le date già calcolate per il report,
# con i punti sostituiti da trattini (più adatti a un nome di file).
if periodo_dal != "n/d":
    dal_file = periodo_dal.replace(".", "-")
    al_file  = periodo_al.replace(".", "-")
    nome_file = f"report_trading_{broker}_{dal_file}_{al_file}_da-file.xlsx"
else:
    nome_file = f"report_trading_{broker}_periodo-n-d_da-file.xlsx"

# --- Salvataggio sul Desktop ---
# Il Desktop è la cartella "Desktop" dentro il profilo utente.
desktop = os.path.join(os.path.expanduser("~"), "Desktop")
percorso_file = os.path.join(desktop, nome_file)
wb.save(percorso_file)

print(f"Report salvato: {percorso_file}")
print(f"  - Broker rilevato: {info_conto['societa']}")
print(f"  - Periodo: dal {periodo_dal} al {periodo_al}")
print(f"  - Operazioni: {len(df_trade)} trade")
print(f"  - Movimenti di cassa: {len(df_cassa)} movimenti")
print("  - IMPORTANTE: aprire e salvare in Excel prima di inviare, per calcolare i totali.")

## APPENDICE - Come leggere il report

Il report riguarda operazioni di trading su strumenti finanziari derivati (CFD)
effettuate tramite broker esteri. Si chiariscono i punti che generano più spesso
confusione.

### "BUY" e "SELL" NON sono acquisti o vendite di beni o valute reali

Le diciture BUY (compra) o SELL (vendi) su strumenti come EURUSD, GBPUSD, USDJPY o
XAUUSD non indicano l'acquisto o la vendita fisica di valuta estera, oro o altre merci.

Si tratta di contratti finanziari (CFD - Contract For Difference): contratti che
replicano la variazione di prezzo di uno strumento. Il bene o la valuta sottostante
non viene mai posseduto. Non vengono ricevuti dollari, sterline, yen o lingotti d'oro:
esiste solo un profitto o una perdita in denaro (in USD), in base all'andamento del prezzo.

- BUY = posizione al rialzo ("lunga"): si punta sull'aumento del prezzo
- SELL = posizione al ribasso ("corta"): si punta sulla diminuzione del prezzo

In entrambi i casi il risultato è solo un importo in denaro (in USD), non un bene fisico.

### Cosa sono le coppie di valute (Forex)

Il Forex (Foreign Exchange) è il mercato dei tassi di cambio. Gli strumenti si
presentano come coppie di valute, es. EURUSD: la prima valuta (EUR) è la "base",
la seconda (USD) è la "quotata".

Operare su una coppia significa puntare sulla variazione del tasso di cambio, non
scambiare fisicamente le due valute. Alcuni esempi di coppie:

- EURUSD - Euro / Dollaro USA
- GBPUSD - Sterlina britannica / Dollaro USA
- USDJPY - Dollaro USA / Yen giapponese
- XAUUSD - Oro / Dollaro USA

### Tutti gli importi sono in dollari USA (USD)

Il conto di trading è denominato in USD. Ogni cifra del report (profitti, perdite,
commissioni) è espressa in dollari. La conversione in euro è a cura dello studio
commercialistico, ai cambi di riferimento.

### Come vengono contate le operazioni

Ogni riga del foglio Operazioni corrisponde a una chiusura, non a una posizione aperta.
I dati vengono letti dalla sezione "Affari" del report MetaTrader (i movimenti di
chiusura del conto) e non dalla sezione "Posizioni", che presenta invece i trade già
aggregati.

La scelta è voluta: quando una posizione viene chiusa in più momenti (chiusure parziali),
la sezione Posizioni la mostra come un'unica riga, mentre gli Affari registrano ogni
chiusura separatamente. Contando le chiusure, ogni realizzo compare come operazione a sé,
con la propria data, prezzo e risultato. Per questo il numero di operazioni del report può
risultare superiore al numero di posizioni aperte, e coincide con il numero di operazioni
indicato nel riepilogo ufficiale di MetaTrader. Il risultato netto complessivo non cambia.

### Broker esteri

Quando i broker hanno sede estera, possono sussistere adempimenti specifici (es. quadro
RW per il monitoraggio delle attività finanziarie detenute all'estero, ed eventuale
imposta di bollo), di competenza dello studio commercialistico.